In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')
import os

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 1. 数据加载和预处理
def load_and_preprocess_data(file_path):
    """加载并预处理数据"""
    print("正在加载数据...")
    df = pd.read_csv(file_path)
    
    # 转换日期时间
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['sitename', 'date'])
    
    # 检查缺失值
    print(f"数据形状: {df.shape}")
    print(f"站点列表: {df['sitename'].unique()}")
    
    return df

# 2. 创建时间序列数据集
def create_sequences(data, site_data, n_past=48, n_future=24):
    """为单个站点创建时间序列数据集"""
    X, y = [], []
    
    # 选择相关特征
    feature_cols = ['aqi', 'pm2.5', 'pm10', 'o3', 'no2', 'so2', 'co', 'windspeed', 'winddirec']
    available_cols = [col for col in feature_cols if col in site_data.columns]
    
    # 确保所有特征列都是数值类型
    for col in available_cols:
        site_data[col] = pd.to_numeric(site_data[col], errors='coerce')
    
    # 填充缺失值
    site_data[available_cols] = site_data[available_cols].fillna(method='ffill').fillna(method='bfill')
    
    # 提取特征
    features = site_data[available_cols].values
    
    for i in range(n_past, len(features) - n_future):
        X.append(features[i-n_past:i])
        # 预测未来24小时的AQI
        y.append(features[i:i+n_future, 0])  # 第一列是AQI
    
    return np.array(X), np.array(y), available_cols

# 3. 数据准备函数
def prepare_data(df, n_past=48, n_future=24, test_ratio=0.2):
    """为所有站点准备数据"""
    sites = df['sitename'].unique()
    site_data_dict = {}
    
    for site in sites:
        print(f"正在处理站点: {site}")
        site_data = df[df['sitename'] == site].copy()
        
        # 创建序列
        X, y, features = create_sequences(df, site_data, n_past, n_future)
        
        if len(X) > 0:
            # 划分训练集和测试集
            split_idx = int(len(X) * (1 - test_ratio))
            
            X_train, X_test = X[:split_idx], X[split_idx:]
            y_train, y_test = y[:split_idx], y[split_idx:]
            
            # 标准化（只在训练集上拟合）
            scalers = {}
            X_train_scaled = np.zeros_like(X_train)
            X_test_scaled = np.zeros_like(X_test)
            
            # 为每个特征维度标准化
            for i in range(X_train.shape[2]):
                scaler = StandardScaler()
                # 重塑为2D以进行标准化
                train_feature = X_train[:, :, i].reshape(-1, 1)
                test_feature = X_test[:, :, i].reshape(-1, 1)
                
                scaler.fit(train_feature)
                X_train_scaled[:, :, i] = scaler.transform(train_feature).reshape(X_train.shape[0], X_train.shape[1])
                X_test_scaled[:, :, i] = scaler.transform(test_feature).reshape(X_test.shape[0], X_test.shape[1])
                
                scalers[i] = scaler
            
            site_data_dict[site] = {
                'X_train': X_train_scaled,
                'X_test': X_test_scaled,
                'y_train': y_train,
                'y_test': y_test,
                'features': features,
                'scalers': scalers,
                'original_data': site_data
            }
    
    return site_data_dict

# 4. 构建LSTM模型
def build_lstm_model(input_shape, n_future=24):
    """构建LSTM模型"""
    model = Sequential([
        Input(shape=input_shape),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(n_future)  # 输出未来24小时的预测
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# 5. 训练模型
def train_models(site_data_dict, n_future=24, epochs=50, batch_size=32):
    """为每个站点训练模型"""
    models = {}
    histories = {}
    
    for site, data in site_data_dict.items():
        print(f"\n正在训练站点 {site} 的模型...")
        
        # 获取输入形状
        input_shape = (data['X_train'].shape[1], data['X_train'].shape[2])
        
        # 构建模型
        model = build_lstm_model(input_shape, n_future)
        
        # 设置早停
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        
        # 训练模型
        history = model.fit(
            data['X_train'], data['y_train'],
            validation_split=0.2,
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stop],
            verbose=0
        )
        
        models[site] = model
        histories[site] = history
        
        print(f"站点 {site} 训练完成")
    
    return models, histories

# 6. 预测和评估
def evaluate_models(models, site_data_dict):
    """评估所有站点的模型"""
    predictions = {}
    metrics_all = {}
    
    for site, data in site_data_dict.items():
        print(f"\n评估站点 {site}...")
        
        # 预测
        y_pred = models[site].predict(data['X_test'], verbose=0)
        
        # 反标准化AQI预测值（如果需要，这里我们预测的是原始AQI值）
        # 注意：在我们的设置中，y_train/y_test已经是原始AQI值，所以不需要反标准化
        
        # 存储预测结果
        predictions[site] = {
            'y_true': data['y_test'],
            'y_pred': y_pred
        }
        
        # 计算整体指标
        y_true_flat = data['y_test'].flatten()
        y_pred_flat = y_pred.flatten()
        
        rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
        mae = mean_absolute_error(y_true_flat, y_pred_flat)
        r2 = r2_score(y_true_flat, y_pred_flat)
        
        # 计算不同时间段的指标
        time_periods = {
            '1-6h': (0, 6),
            '7-12h': (6, 12),
            '13-18h': (12, 18),
            '19-24h': (18, 24)
        }
        
        period_metrics = {}
        for period, (start, end) in time_periods.items():
            period_true = data['y_test'][:, start:end].flatten()
            period_pred = y_pred[:, start:end].flatten()
            
            period_rmse = np.sqrt(mean_squared_error(period_true, period_pred))
            period_mae = mean_absolute_error(period_true, period_pred)
            period_r2 = r2_score(period_true, period_pred)
            
            period_metrics[period] = {
                'RMSE': period_rmse,
                'MAE': period_mae,
                'R²': period_r2
            }
        
        metrics_all[site] = {
            'overall': {'RMSE': rmse, 'MAE': mae, 'R²': r2},
            'periods': period_metrics
        }
        
        print(f"整体指标 - RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}")
    
    return predictions, metrics_all

# 7. 可视化结果
def visualize_results(predictions, metrics_all, site_data_dict):
    """可视化预测结果和性能指标"""
    
    # 一、特定样本的时间序列拟合情况
    print("\n一、时间序列拟合情况可视化")
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    sites = list(predictions.keys())
    sample_idx = 0  # 选择第一个测试样本
    
    for i, site in enumerate(sites[:6]):  # 最多显示6个站点
        if i < len(axes):
            y_true_sample = predictions[site]['y_true'][sample_idx]
            y_pred_sample = predictions[site]['y_pred'][sample_idx]
            
            hours = range(1, 25)
            ax = axes[i]
            ax.plot(hours, y_true_sample, 'b-', label='实际AQI值', linewidth=2)
            ax.plot(hours, y_pred_sample, 'r--', label='模型预测AQI值', linewidth=2)
            ax.set_xlabel('预测时长 (小时)', fontsize=10)
            ax.set_ylabel('AQI值', fontsize=10)
            ax.set_title(f'{site}站点 - 预测对比', fontsize=12)
            ax.legend()
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('站点AQI预测对比.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 二、不同时间段性能表现
    print("\n二、不同时间段性能表现")
    # 创建DataFrame用于存储所有站点的性能指标
    metrics_df_list = []
    
    for site in sites:
        for period in ['1-6h', '7-12h', '13-18h', '19-24h']:
            metrics = metrics_all[site]['periods'][period]
            metrics_df_list.append({
                '站点': site,
                '时间段': period,
                'RMSE': metrics['RMSE'],
                'MAE': metrics['MAE'],
                'R²': metrics['R²']
            })
    
    metrics_df = pd.DataFrame(metrics_df_list)
    
    # 绘制RMSE和MAE的热力图
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # RMSE热力图
    rmse_pivot = metrics_df.pivot_table(index='站点', columns='时间段', values='RMSE')
    sns.heatmap(rmse_pivot, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0, 0])
    axes[0, 0].set_title('不同站点和预测时段的RMSE', fontsize=14)
    
    # MAE热力图
    mae_pivot = metrics_df.pivot_table(index='站点', columns='时间段', values='MAE')
    sns.heatmap(mae_pivot, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0, 1])
    axes[0, 1].set_title('不同站点和预测时段的MAE', fontsize=14)
    
    # 绘制折线图展示变化趋势
    for site in sites[:3]:  # 显示前3个站点的趋势
        site_data = metrics_df[metrics_df['站点'] == site]
        axes[1, 0].plot(site_data['时间段'], site_data['RMSE'], marker='o', label=site, linewidth=2)
        axes[1, 1].plot(site_data['时间段'], site_data['MAE'], marker='s', label=site, linewidth=2)
    
    axes[1, 0].set_xlabel('预测时段', fontsize=12)
    axes[1, 0].set_ylabel('RMSE', fontsize=12)
    axes[1, 0].set_title('RMSE随预测时长的变化', fontsize=14)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].set_xlabel('预测时段', fontsize=12)
    axes[1, 1].set_ylabel('MAE', fontsize=12)
    axes[1, 1].set_title('MAE随预测时长的变化', fontsize=14)
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('不同时段性能表现.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 三、R²值分布
    print("\n三、R²值分布可视化")
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for i, site in enumerate(sites[:6]):
        if i < len(axes):
            r2_values = []
            periods = []
            
            for period in ['1-6h', '7-12h', '13-18h', '19-24h']:
                r2_values.append(metrics_all[site]['periods'][period]['R²'])
                periods.append(period)
            
            ax = axes[i]
            bars = ax.bar(periods, r2_values, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
            ax.set_ylim([0, 1])
            ax.set_xlabel('预测时段', fontsize=10)
            ax.set_ylabel('R²值', fontsize=10)
            ax.set_title(f'{site}站点 - 不同时段R²值', fontsize=12)
            ax.grid(True, alpha=0.3, axis='y')
            
            # 在柱状图上添加数值
            for bar, value in zip(bars, r2_values):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                       f'{value:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('不同时段R2值分布.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return metrics_df

# 8. 生成综合报告
def generate_report(metrics_df, predictions, site_data_dict):
    """生成综合评估报告"""
    print("\n" + "="*80)
    print("LSTM多站点AQI预测模型评估报告")
    print("="*80)
    
    # 整体性能
    print("\n一、整体性能摘要:")
    print("-"*50)
    
    overall_r2 = []
    for site in predictions.keys():
        overall_r2.append(metrics_df[metrics_df['站点'] == site]['R²'].mean())
    
    print(f"平均R²值: {np.mean(overall_r2):.4f}")
    print(f"最佳R²值: {np.max(overall_r2):.4f} (站点: {list(predictions.keys())[np.argmax(overall_r2)]})")
    print(f"最差R²值: {np.min(overall_r2):.4f} (站点: {list(predictions.keys())[np.argmin(overall_r2)]})")
    
    # 各站点详细性能
    print("\n二、各站点详细性能:")
    print("-"*50)
    
    for site in predictions.keys():
        site_metrics = metrics_df[metrics_df['站点'] == site]
        avg_rmse = site_metrics['RMSE'].mean()
        avg_mae = site_metrics['MAE'].mean()
        avg_r2 = site_metrics['R²'].mean()
        
        print(f"\n{site}站点:")
        print(f"  平均RMSE: {avg_rmse:.2f}")
        print(f"  平均MAE: {avg_mae:.2f}")
        print(f"  平均R²: {avg_r2:.4f}")
        
        # 性能随时间变化
        r2_trend = list(site_metrics.sort_values('时间段')['R²'])
        if len(r2_trend) == 4:
            trend = "下降" if r2_trend[0] > r2_trend[-1] else "上升" if r2_trend[0] < r2_trend[-1] else "稳定"
            print(f"  R²趋势 (1-6h到19-24h): {trend}")
    
    # 模型应用建议
    print("\n三、模型应用建议:")
    print("-"*50)
    
    # 分析短期和长期预测性能
    early_period_r2 = metrics_df[metrics_df['时间段'].isin(['1-6h', '7-12h'])]['R²'].mean()
    late_period_r2 = metrics_df[metrics_df['时间段'].isin(['13-18h', '19-24h'])]['R²'].mean()
    
    if early_period_r2 > 0.7 and late_period_r2 > 0.5:
        print("✓ 模型适用于短期和长期预警")
        print("  建议：可用于未来24小时空气质量预报系统")
    elif early_period_r2 > 0.7:
        print("✓ 模型主要适用于短期预警（1-12小时）")
        print("  建议：重点用于临近空气质量预报")
    elif late_period_r2 > 0.5:
        print("✓ 模型在长期预测中表现稳定")
        print("  建议：可用于趋势性空气质量预警")
    else:
        print("⚠ 模型性能有待提升")
        print("  建议：考虑增加特征或调整模型参数")
    
    print("\n" + "="*80)

# 主函数
def main():
    """主函数"""
    # 文件路径
    file_path = r"F:\机器学习\kaohsiung_5sites_final.csv"
    
    # 检查文件是否存在
    if not os.path.exists(file_path):
        print(f"错误：文件 {file_path} 不存在！")
        print("请检查文件路径是否正确。")
        # 使用当前目录下的文件作为示例
        print("将使用示例数据模式...")
        return
    
    try:
        # 1. 加载数据
        df = load_and_preprocess_data(file_path)
        
        # 2. 准备数据
        print("\n正在准备时间序列数据...")
        site_data_dict = prepare_data(df, n_past=48, n_future=24, test_ratio=0.2)
        
        if len(site_data_dict) == 0:
            print("错误：无法从数据中创建有效的时间序列！")
            return
        
        # 3. 训练模型
        print("\n开始训练LSTM模型...")
        models, histories = train_models(
            site_data_dict, 
            n_future=24, 
            epochs=50, 
            batch_size=32
        )
        
        # 4. 评估模型
        print("\n正在评估模型性能...")
        predictions, metrics_all = evaluate_models(models, site_data_dict)
        
        # 5. 可视化结果
        print("\n正在生成可视化结果...")
        metrics_df = visualize_results(predictions, metrics_all, site_data_dict)
        
        # 6. 生成报告
        generate_report(metrics_df, predictions, site_data_dict)
        
        # 7. 保存结果
        print("\n正在保存结果...")
        metrics_df.to_csv('各站点预测性能.csv', index=False, encoding='utf-8-sig')
        
        # 保存模型预测结果
        predictions_df_list = []
        for site, pred_data in predictions.items():
            for i in range(min(10, len(pred_data['y_true']))):  # 保存前10个样本
                for hour in range(24):
                    predictions_df_list.append({
                        '站点': site,
                        '样本': i,
                        '预测时长(小时)': hour + 1,
                        '实际AQI值': pred_data['y_true'][i, hour],
                        '预测AQI值': pred_data['y_pred'][i, hour]
                    })
        
        predictions_df = pd.DataFrame(predictions_df_list)
        predictions_df.to_csv('预测结果示例.csv', index=False, encoding='utf-8-sig')
        
        print("\n所有任务完成！")
        print("已生成的文件:")
        print("1. 站点AQI预测对比.png - 时间序列拟合图")
        print("2. 不同时段性能表现.png - 性能指标热力图和趋势图")
        print("3. 不同时段R2值分布.png - R²值分布图")
        print("4. 各站点预测性能.csv - 详细性能指标")
        print("5. 预测结果示例.csv - 具体预测值示例")
        
    except Exception as e:
        print(f"程序执行出错: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

正在加载数据...
数据形状: (223971, 23)
站点列表: ['Linyuan' 'Meinong' 'Qianjin' 'Xiaogang' 'Zuoying']

正在准备时间序列数据...
正在处理站点: Linyuan
正在处理站点: Meinong
正在处理站点: Qianjin
正在处理站点: Xiaogang
正在处理站点: Zuoying

开始训练LSTM模型...

正在训练站点 Linyuan 的模型...
站点 Linyuan 训练完成

正在训练站点 Meinong 的模型...


KeyboardInterrupt: 